# v7: tracked line paths and gradual corner alignment

Based on v6, with changes for both marked and unmarked sharp corners. Camera only: no measured angle or guaranteed branch identity. Keep v4 and v6 for comparison. Run one cell at a time; driving remains commented out. Copy your reliable motor/threshold calibration first.

- Track plausible paths across bands and recent frames rather than choosing the largest blob in each band. Orange dots are candidates, green dots the selected path. Broad horizontal slices are treated as missing observations. Similar competing paths cause a brief stationary HOLD, then a latched stop if unresolved.
- Ordinary sharp corners steer gradually as the local line returns; visible opposite bends can change steering without a locked pivot. Missing-line recovery retains the last direction with bounded deadlines. Straight-line gap behavior is retained.
- Green corners commit with one wheel stationary, then capture and align with nonnegative proportional wheel commands. Completion uses local position/direction over multiple frames, not a rotation timer. Timeout remains a failure bound.
- Short-lived marker identities suppress the handled marker and allow a different marker without a global cooldown or marker-free interval. A new track first appearing below the trigger is conservatively ignored, except on the first processed frame. Show the approach to each marker. Identity association can still fail during fast rotation, occlusion or crowded markers.
- BOTH remains an ambiguous-instruction stop. Fourth band is enabled at 20–34% of image height. It contributes to path tracking and has a small steering weight; near/middle still determine turn completion.

Defaults are trial settings. Camera perspective, tape width, motor stall speed and a stationary-wheel arc need floor testing. A zero motor command is not a mechanical brake. Software cannot protect against a frozen driver/process.

## 1. Initialize camera once


In [ ]:
import time
import cv2 #카메라
import numpy as np # 
import traitlets #값 바뀔 때 신호 연동
import ipywidgets as widgets #화면 위젯

from IPython.display import display #
from jetbot import Camera, Robot, bgr8_to_jpeg

robot = Robot()
robot.stop()

camera = Camera.instance(width=224, height=224)

live_view = widgets.Image(format="jpeg", width=336, height=336) #camera가 보여주는 영상 만들 공간

camera_link = traitlets.dlink(
    (camera, "value"),
    (live_view, "value"),
    transform=bgr8_to_jpeg
)

display(live_view)


In [ ]:
latest = {"sample": None} #가장 최근에 카메라가 인식했던 영상 사진 

def receive_frame(change): #
    latest["sample"] = (change["new"], time.monotonic()) #새 장면 찍힐 때 마다 이미지와 시간

camera.observe(receive_frame, names="value") #카메라가 새로운 장면 보일 때 마다 리시브 프레임 실행해서 저장

time.sleep(0.5) #카메라 콜백 시간

if latest["sample"] is None:
    print("No camera frame yet.")
else:
    print("Camera frame received.")


## 2. Black line and green detection
Green mask should show only physical tape markers. Drawn overlay dots are never fed into detection. Tune HSV, area bounds, and trigger position while stationary.


In [ ]:
BLACK_THRESHOLD = 70
EXTRA_FAR_ENABLED = True
BANDS = [('near', .72, .88), ('middle', .54, .70), ('far', .36, .52)]
if EXTRA_FAR_ENABLED:
    BANDS.append(('extra_far', .20, .34))
LINE_WEIGHTS = {'near': .65, 'middle': .25, 'far': .10, 'extra_far': .05}
TRACK_MEMORY = .35
TRACK_MAX_JUMP = .65
TRACK_AMBIGUITY = .08

class LineTracker:
    """Small beam search over band candidates; never invent missing observations."""
    def __init__(self):
        self.previous = {}
        self.last_seen = None
        self.ambiguous = False
        self.confidence = 0.0

    def select(self, candidates, now, preferred=0):
        names = [name for name, _, _ in BANDS]
        history = self.previous if self.last_seen is not None and now-self.last_seen <= TRACK_MEMORY else {}
        # Each candidate is (normalized x, connected-component label).
        paths = [(0., [])]
        for name in names:
            options = candidates.get(name, [])
            expanded = []
            for cost, path in paths:
                # Allow a missing band rather than forcing a cross-branch jump.
                expanded.append((cost + .8, path + [None]))
                for x, label in options:
                    prior = next((v for v in reversed(path) if v is not None), None)
                    if prior is not None:
                        distance = abs(x-prior[0])
                        if distance > TRACK_MAX_JUMP:
                            continue
                        # Same component is supporting evidence, not proof at intersections.
                        if label != prior[1] and distance > .28:
                            continue
                        continuity = .5*distance + (.18 if label != prior[1] else 0.)
                    else:
                        continuity = .12*abs(x)
                    temporal = .35*abs(x-history[name]) if name in history else 0.
                    # Marker direction biases competing branches, not the returned offsets.
                    preference = -.30*preferred*x if preferred and name != 'near' else 0.
                    expanded.append((cost + continuity + temporal + preference, path+[(x,label)]))
            paths = sorted(expanded, key=lambda p:p[0])[:24]
        best_cost, best = paths[0]
        self.ambiguous = False
        for cost, path in paths[1:]:
            distinct = any(a is not None and b is not None and abs(a[0]-b[0]) > .25
                           for a,b in zip(best,path))
            if distinct and cost-best_cost < TRACK_AMBIGUITY:
                self.ambiguous = True
                break
        selected = {name: (item[0] if item is not None else None) for name,item in zip(names,best)}
        self.confidence = sum(v is not None for v in selected.values()) / float(len(names))
        if self.ambiguous:
            self.confidence = 0.
            return dict.fromkeys(names)
        if any(v is not None for v in selected.values()):
            self.previous = {k:v for k,v in selected.items() if v is not None}
            self.last_seen = now
        return selected

    def detect(self, frame, now, preferred=0):
        height, width = frame.shape[:2]
        gray = cv2.GaussianBlur(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (5,5), 0)
        _, mask = cv2.threshold(gray, BLACK_THRESHOLD, 255, cv2.THRESH_BINARY_INV)
        _, labels = cv2.connectedComponents(mask, connectivity=8)
        candidates = {}
        dark_evidence = False
        overlay = frame.copy()
        for name, top, bottom in BANDS:
            y0,y1 = int(top*height),int(bottom*height)
            cy = (y0+y1)//2
            # Narrow central slice preserves separate branches inside a wider band.
            occupied = np.mean(mask[max(y0,cy-2):min(y1,cy+3),:] > 0, axis=0) >= .6
            edges = np.diff(np.r_[False,occupied,False].astype(np.int8))
            starts, ends = np.where(edges==1)[0], np.where(edges==-1)[0]
            dark_evidence = dark_evidence or bool(np.any(occupied))
            choices = []
            for start,end in zip(starts,ends):
                if not 2 <= end-start <= width*.55:
                    continue  # broad crossing/dark patch is not a reliable centerline
                cx = (start+end-1)/2.
                component_values = labels[max(y0,cy-2):min(y1,cy+3),start:end]
                valid = component_values[component_values > 0]
                if valid.size:
                    label = int(np.bincount(valid).argmax())
                    choices.append(((cx-width/2.)/(width/2.),label))
                    cv2.circle(overlay,(int(cx),cy),3,(0,165,255),1)
            # Bound work at a noisy frame; overflow is explicitly uncertain.
            candidates[name] = choices[:8]
            cv2.rectangle(overlay,(0,y0),(width-1,y1),(255,0,0),1)
            cv2.putText(overlay,name,(3,y0+12),cv2.FONT_HERSHEY_SIMPLEX,.35,(0,0,255),1)
        positions = self.select(candidates,now,preferred)
        if (any(len(v) >= 8 for v in candidates.values()) or
                (dark_evidence and not any(v is not None for v in positions.values()))):
            self.ambiguous = True
            self.confidence = 0.
            positions = dict.fromkeys(positions)
        for name,top,bottom in BANDS:
            if positions[name] is not None:
                cv2.circle(overlay,(int(width*(1+positions[name])/2),int(height*(top+bottom)/2)),4,(0,255,0),-1)
        cv2.line(overlay,(width//2,0),(width//2,height-1),(0,255,255),1)
        return positions,overlay,mask

# OpenCV HSV hue range is 0..179. Tune these using the actual green tape.
GREEN_LOWER = np.array([35, 70, 50], dtype=np.uint8)
GREEN_UPPER = np.array([90, 255, 255], dtype=np.uint8)
GREEN_MIN_AREA = 40
GREEN_MAX_AREA = 3000
GREEN_ROI_TOP = 0.45
GREEN_ROI_BOTTOM = 0.92
GREEN_TRIGGER_Y = 0.60  # Require a nearby marker before initiating a turn.
GREEN_SIDE_MARGIN = 8.0
GREEN_MAX_SIDE_DISTANCE = 65.0  # Pixels, at the configured 224px resolution.
GREEN_PAIR_Y_TOLERANCE = 0.15

def detect_green(frame):
    height, width = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, GREEN_LOWER, GREEN_UPPER)
    mask[:int(height * GREEN_ROI_TOP), :] = 0
    mask[int(height * GREEN_ROI_BOTTOM):, :] = 0
    kernel = np.ones((3, 3), dtype=np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[-2]
    markers = []
    for contour in contours:
        area = cv2.contourArea(contour)
        x, y, w, h = cv2.boundingRect(contour)
        if not GREEN_MIN_AREA <= area <= GREEN_MAX_AREA or h == 0:
            continue
        if not 0.4 <= w / float(h) <= 2.5:
            continue
        moments = cv2.moments(contour)
        if moments['m00'] <= 0:
            continue
        markers.append(dict(x=moments['m10']/moments['m00'],
                            y=moments['m01']/moments['m00'],
                            area=area, w=w, h=h, box=(x,y,w,h)))
    markers.sort(key=lambda item: item['area'], reverse=True)
    return markers, mask

def get_line_points(positions, width, height, include_extra=False):
    band_y = {name: (top + bottom) / 2.0 for name, top, bottom in BANDS}
    # Keep marker-side geometry local; include every band in the path overlay.
    names = [name for name, _, _ in reversed(BANDS)] if include_extra else ('far', 'middle', 'near')
    return [(width * (1.0 + positions[name]) / 2.0, height * band_y[name])
            for name in names if positions.get(name) is not None]

def classify_green_direction(positions, markers, width, height):
    if not markers:
        return 'NONE'
    points = get_line_points(positions, width, height)
    if len(points) < 2:
        return 'UNKNOWN'
    near_x, near_y = points[-1]
    dx, dy = points[0][0] - near_x, points[0][1] - near_y
    length = (dx * dx + dy * dy) ** 0.5
    if length < 1e-6:
        return 'UNKNOWN'
    dx, dy = dx / length, dy / length
    sides = {'LEFT': [], 'RIGHT': []}
    for marker in markers:
        if marker['y'] < height * GREEN_TRIGGER_Y:
            continue
        score = (marker['x'] - near_x) * dy - (marker['y'] - near_y) * dx
        if GREEN_SIDE_MARGIN < abs(score) <= GREEN_MAX_SIDE_DISTANCE:
            sides['LEFT' if score > 0 else 'RIGHT'].append(marker)
    if sides['LEFT'] and sides['RIGHT']:
        paired = any(abs(a['y'] - b['y']) <= height * GREEN_PAIR_Y_TOLERANCE
                     for a in sides['LEFT'] for b in sides['RIGHT'])
        return 'BOTH' if paired else 'UNKNOWN'
    if sides['LEFT']:
        return 'LEFT'
    if sides['RIGHT']:
        return 'RIGHT'
    return 'CENTER'

MARKER_MATCH_DISTANCE = .18  # Fraction of frame diagonal-axis scale (max width/height).
MARKER_MEMORY = 2.0

class MarkerTracker:
    """Short-lived visual identities. New low-entering blobs are not assumed new markers."""
    def __init__(self):
        self.tracks = {}
        self.next_id = 1
        self.initial = True

    def update(self,markers,now,width,height):
        self.tracks = {key:t for key,t in self.tracks.items() if now-t['seen']<=MARKER_MEMORY}
        scale = float(max(width,height))
        pairs = []
        for i,m in enumerate(markers):
            for key,t in self.tracks.items():
                old = t['marker']
                distance = ((m['x']-old['x'])**2+(m['y']-old['y'])**2)**.5/scale
                ratio = m['area']/max(old['area'],1.)
                if distance<=MARKER_MATCH_DISTANCE and .3<=ratio<=3.3:
                    pairs.append((distance,i,key))
        matched,used = {},set()
        for _,i,key in sorted(pairs):
            if i not in matched and key not in used:
                matched[i]=key
                used.add(key)
        result = []
        for i,m in enumerate(markers):
            key = matched.get(i)
            if key is None:
                key = self.next_id
                self.next_id += 1
                self.tracks[key] = dict(handled=False,approached=(self.initial or m['y']<height*GREEN_TRIGGER_Y))
            t = self.tracks[key]
            t['approached'] = t['approached'] or m['y']<height*GREEN_TRIGGER_Y
            t.update(marker=dict(m),seen=now)
            item = dict(m)
            item.update(track_id=key,eligible=t['approached'] and not t['handled'])
            result.append(item)
        self.initial = False
        return result

    def consume(self,ids):
        for key in ids:
            if key in self.tracks:
                self.tracks[key]['handled'] = True

    def instruction(self,positions,markers,width,height):
        ready = []
        for m in markers:
            if m['eligible'] and classify_green_direction(positions,[m],width,height) in ('LEFT','RIGHT'):
                ready.append(m)
        direction = classify_green_direction(positions,ready,width,height)
        return direction,tuple(sorted(m['track_id'] for m in ready))


## 3. Ordinary line and sharp-corner controller
Near observations have the largest weight. Local near/middle alignment controls corner capture; far observations anticipate the next bend during following. During visible recovery the steering sign can change for an S-bend. No wheel reverses. All turn/search deadlines remain bounded and latched on failure.


In [ ]:
# Trial motor values; copy the robot's reliable calibration before driving.
BASE_SPEED = .15
CURVE_SPEED = .13
KP = .14
MAX_SPEED = .25
TURN_SPEED = .15
TURN_ENTER = .55
TURN_EXIT = .22
TURN_TIMEOUT = 1.0
LOST_TIMEOUT = .35
FRAME_TIMEOUT = .5
ALIGN_SPREAD = .12
ALIGN_CONFIRM_FRAMES = 2
UNCERTAIN_TIMEOUT = .35

def clamp(value, low, high):
    return max(low,min(high,value))

def local_aligned(positions):
    near,middle = positions.get('near'),positions.get('middle')
    return (near is not None and middle is not None and
            abs(near)<TURN_EXIT and abs(middle)<TURN_EXIT and abs(middle-near)<=ALIGN_SPREAD)

def steering_error(positions):
    visible = [(name,v) for name,v in positions.items() if v is not None and name in LINE_WEIGHTS]
    if not visible:
        return None
    error = sum(LINE_WEIGHTS[n]*v for n,v in visible)/sum(LINE_WEIGHTS[n] for n,_ in visible)
    near,middle = positions.get('near'),positions.get('middle')
    if near is not None and middle is not None:
        error += .20*(middle-near)
    return clamp(error,-1.,1.)

def alignment_commands(positions, speed=TURN_SPEED):
    # Local position dominates capture; far band may already contain the NEXT bend.
    near,middle = positions.get('near'),positions.get('middle')
    if near is not None and middle is not None:
        error = clamp(.8*near+.2*middle+.2*(middle-near),-1.,1.)
    else:
        error = steering_error(positions)
    if error is None:
        return 0.,0.
    outer = clamp(speed,0.,MAX_SPEED)
    strength = min(abs(error)/TURN_ENTER,1.)
    inner = outer*(1.-strength)
    return (outer,inner) if error>=0 else (inner,outer)

class CurveController:
    def __init__(self):
        self.turn_direction = 0
        self.turn_started = None
        self.lost_started = None
        self.last_direction = 0
        self.aligned_frames = 0
        self.stopped = False

    def stop(self,reason):
        self.stopped = True
        return 0.,0.,'STOP: '+reason

    def pivot(self,direction):
        speed = clamp(TURN_SPEED,0.,MAX_SPEED)
        return (speed,0.) if direction>0 else (0.,speed)

    def step(self,positions,now):
        if self.stopped:
            return 0.,0.,'STOP: controller stopped'
        if self.turn_started is not None and now-self.turn_started>=TURN_TIMEOUT:
            return self.stop('turn timed out')
        error = steering_error(positions)
        if error is None:
            self.aligned_frames = 0
            if self.lost_started is None:
                self.lost_started = now
            if now-self.lost_started>=LOST_TIMEOUT:
                return self.stop('line not reacquired')
            if not self.last_direction:
                return self.stop('line lost without a known turn direction')
            if self.turn_started is None:
                self.turn_started = now
            self.turn_direction = self.last_direction
            left,right = self.pivot(self.turn_direction)
            return left,right,'SEARCH'
        self.lost_started = None
        if abs(error)>.15:
            self.last_direction = 1 if error>0 else -1
        if self.turn_started is None and abs(error)>=TURN_ENTER:
            self.turn_started = now
            self.turn_direction = 1 if error>0 else -1
        if self.turn_started is not None:
            self.aligned_frames = self.aligned_frames+1 if local_aligned(positions) else 0
            if self.aligned_frames>=ALIGN_CONFIRM_FRAMES:
                self.turn_started = None
                self.turn_direction = 0
                self.aligned_frames = 0
            else:
                # Recompute from observations, including opposite bends; only blind search
                # commits to the remembered direction. Never keep pivoting through alignment.
                if positions.get('near') is not None and positions.get('middle') is not None:
                    left,right = alignment_commands(positions)
                    return left,right,'CURVE_ALIGN'
                left,right = alignment_commands(positions)
                return left,right,'CURVE_CAPTURE'
        bend = min(abs(error)/TURN_ENTER,1.)
        base = BASE_SPEED+(CURVE_SPEED-BASE_SPEED)*bend
        # Slow on partial visibility instead of charging toward an unseen near line.
        if positions.get('near') is None:
            base = CURVE_SPEED
        correction = KP*error
        return clamp(base+correction,0.,MAX_SPEED),clamp(base-correction,0.,MAX_SPEED),'FOLLOW'


## 4. Gap controller
Copy your calibrated gap timeout and speed here.


In [ ]:
# Initial trial values; calibrate on the real floor at your working motor power.
GAP_SPEED = BASE_SPEED
GAP_TIMEOUT = 1.0       # Maximum time driving without confirmed near/middle tape.
GAP_STEER_LIMIT = 0.015 # Only a small remembered correction; not heading feedback.
GAP_APPROACH_FRAMES = 3
GAP_APPROACH_MAX_AGE = 0.6
GAP_ALIGNMENT = 0.25   # Normalized offset from image centre.
GAP_STRAIGHT_SPREAD = 0.12
GAP_CONFIRM_FRAMES = 2

class GapCurveController(CurveController):
    def __init__(self):
        super().__init__()
        self.straight_frames = 0
        self.straight_seen = None
        self.saved_correction = 0.0
        self.gap_started = None
        self.confirm_frames = 0

    def gap_commands(self):
        return (clamp(GAP_SPEED + self.saved_correction, 0.0, MAX_SPEED),
                clamp(GAP_SPEED - self.saved_correction, 0.0, MAX_SPEED),
                "GAP: forward, awaiting tape")

    def step(self, positions, now):
        if self.stopped:
            return 0.0, 0.0, "STOP: controller stopped"
        near = positions.get("near")
        middle = positions.get("middle")
        values = [value for value in positions.values() if value is not None]

        if self.gap_started is not None:
            # Never reset this deadline because of an isolated noisy detection.
            if now - self.gap_started >= GAP_TIMEOUT:
                return self.stop("gap crossing timed out")
            if near is not None and middle is not None:
                self.confirm_frames += 1
            else:
                self.confirm_frames = 0
            if self.confirm_frames < GAP_CONFIRM_FRAMES:
                return self.gap_commands()
            self.gap_started = None
            self.confirm_frames = 0
            self.lost_started = None
            self.last_direction = 0
            # Resume the existing controller using the newly observed tape.

        if not values:
            recent_straight = (self.straight_seen is not None and
                               now - self.straight_seen <= GAP_APPROACH_MAX_AGE)
            if (self.turn_started is None and recent_straight and
                    self.straight_frames >= GAP_APPROACH_FRAMES):
                self.gap_started = now
                self.straight_frames = 0
                self.straight_seen = None
                self.confirm_frames = 0
                return self.gap_commands()
            return super().step(positions, now)

        left, right, state = super().step(positions, now)
        aligned = max(abs(value) for value in values) <= GAP_ALIGNMENT
        straight = len(values) >= 2 and max(values) - min(values) <= GAP_STRAIGHT_SPREAD
        if state == "FOLLOW" and near is not None and aligned and straight:
            if self.straight_seen is None or now - self.straight_seen > GAP_APPROACH_MAX_AGE:
                self.straight_frames = 0
            self.straight_frames += 1
            self.straight_seen = now
            correction = clamp((left - right) / 2.0, -GAP_STEER_LIMIT, GAP_STEER_LIMIT)
            self.saved_correction = 0.5 * self.saved_correction + 0.5 * correction
        elif state != "FOLLOW" or not aligned or (len(values) >= 2 and not straight):
            self.straight_frames = 0
            self.straight_seen = None
        # A short single-band transition can occur as the line exits the image.
        # Preserve earlier evidence only until GAP_APPROACH_MAX_AGE expires.
        return left, right, state


## 5. Green corner controller
STOP → committed TURN → gradual ALIGN → FOLLOW. The original turn deadline covers both TURN and ALIGN, including temporary loss. Stable departure prevents immediate acceptance of the old centered line. A marker instruction biases the path selector toward that side, but this is a heuristic, not intersection topology recognition. Marker IDs replace global rearming. A new confirmed marker may take over an ordinary curve, but not a gap or unfinished green maneuver.


In [ ]:
GREEN_CONFIRM_FRAMES = 3
GREEN_STOP_TIME = .12
GREEN_TURN_SPEED = .15
GREEN_TURN_TIMEOUT = 1.5
GREEN_DEPART_FRAMES = 3
GREEN_CAPTURE_FRAMES = 2
GREEN_REACQUIRE_FRAMES = 3

class GreenNavigator:
    def __init__(self):
        self.line = GapCurveController()
        self.state = 'FOLLOW'
        self.state_started = None
        self.direction = None
        self.candidate = None
        self.candidate_frames = 0
        self.consumed_ids = ()
        self.departed = False
        self.depart_frames = 0
        self.capture_frames = 0
        self.reacquire_frames = 0
        self.turn_started = None
        self.uncertain_started = None
        self.stopped = False
        self.reason = ''

    def stop(self,reason):
        self.stopped = True
        self.reason = reason
        return 0.,0.,'STOP: '+reason

    def preferred_direction(self):
        if self.state in ('GREEN_STOP','GREEN_TURN','GREEN_ALIGN'):
            return 1 if self.direction=='RIGHT' else -1
        return 0

    def turn(self):
        speed = clamp(GREEN_TURN_SPEED,0.,MAX_SPEED)
        return (speed,0.,'GREEN_TURN_RIGHT') if self.direction=='RIGHT' else (0.,speed,'GREEN_TURN_LEFT')

    def step(self,positions,direction,now,marker_ids=(),reliable=True):
        self.consumed_ids = ()
        if self.stopped:
            return self.stop(self.reason)
        # Wall-clock deadlines remain active even during ambiguous observations.
        if self.turn_started is not None and now-self.turn_started>=GREEN_TURN_TIMEOUT:
            return self.stop('green turn timed out')
        if not reliable:
            self.line.straight_frames = 0
            self.line.straight_seen = None
            self.candidate = None
            self.candidate_frames = self.capture_frames = self.reacquire_frames = self.depart_frames = 0
            if self.uncertain_started is None:
                self.uncertain_started = now
            if now-self.uncertain_started>=UNCERTAIN_TIMEOUT:
                return self.stop('ambiguous line path')
            return 0.,0.,'HOLD: ambiguous line path'
        self.uncertain_started = None
        if self.state=='GREEN_STOP':
            if now-self.state_started<GREEN_STOP_TIME:
                return 0.,0.,'GREEN_STOP'
            self.departed = False
            self.depart_frames = self.capture_frames = self.reacquire_frames = 0
            self.turn_started = now
            self.state = 'GREEN_TURN'
        if self.state in ('GREEN_TURN','GREEN_ALIGN'):
            if not self.departed:
                self.depart_frames = self.depart_frames+1 if not local_aligned(positions) else 0
                self.departed = self.depart_frames>=GREEN_DEPART_FRAMES
            near,middle = positions.get('near'),positions.get('middle')
            observed = near is not None and middle is not None
            sign = 1 if self.direction=='RIGHT' else -1
            # Do not steer back toward the old line as it slides away on the
            # opposite side of the image during rotation. Capture on the chosen
            # side (or near center), with a plausible local direction.
            capturable = (observed and sign*near>=-.10 and abs(near)<=.65 and
                          abs(middle-near)<=.30)
            if self.departed and capturable:
                self.capture_frames += 1
            else:
                self.capture_frames = 0
            if self.departed and self.capture_frames>=GREEN_CAPTURE_FRAMES:
                self.state = 'GREEN_ALIGN'
            if self.state=='GREEN_ALIGN' and not observed:
                # Return to committed direction while blind; preserve original deadline.
                self.state = 'GREEN_TURN'
                self.reacquire_frames = 0
            if self.state=='GREEN_ALIGN':
                self.reacquire_frames = self.reacquire_frames+1 if local_aligned(positions) else 0
                if self.reacquire_frames>=GREEN_REACQUIRE_FRAMES:
                    self.line = GapCurveController()
                    self.state = 'FOLLOW'
                    self.turn_started = None
                    left,right,_ = self.line.step(positions,now)
                    return left,right,'GREEN_DONE'
                left,right = alignment_commands(positions,GREEN_TURN_SPEED)
                return left,right,'GREEN_ALIGN'
            return self.turn()
        # Identities replace global cooldown. A distinct marker can trigger immediately
        # after the current maneuver. A confirmed marker can also take over a curve,
        # but not an active gap crossing.
        eligible = self.line.gap_started is None and direction in ('LEFT','RIGHT','BOTH') and bool(marker_ids)
        if eligible:
            candidate = (direction,tuple(sorted(marker_ids)))
            if candidate==self.candidate:
                self.candidate_frames += 1
            else:
                self.candidate,self.candidate_frames = candidate,1
            if self.candidate_frames>=GREEN_CONFIRM_FRAMES:
                if direction=='BOTH':
                    return self.stop('BOTH markers: course rule required')
                self.direction = direction
                self.consumed_ids = tuple(marker_ids)
                self.candidate = None
                self.candidate_frames = 0
                self.state,self.state_started = 'GREEN_STOP',now
                return 0.,0.,'GREEN_STOP'
        else:
            self.candidate = None
            self.candidate_frames = 0
        left,right,state = self.line.step(positions,now)
        if self.line.stopped:
            return self.stop(state)
        return left,right,state


## 6. Runner
Preview computes commands but only stops motors. Every run creates fresh controller state. Camera frames are checked for freshness even when no new frame arrives.


In [ ]:
overlay_view = widgets.Image(format='jpeg', width=336, height=336)
mask_view = widgets.Image(format='jpeg', width=336, height=336)
status_view = widgets.Label(value='Ready for stationary preview')
display(widgets.VBox([widgets.HBox([overlay_view, mask_view]), status_view]))

def run_v7(seconds=10, drive=False):
    robot.stop()
    controller = GreenNavigator()
    line_tracker = LineTracker()
    marker_tracker = MarkerTracker()
    deadline = time.monotonic() + seconds
    previous_stamp = None
    next_display = 0.0
    try:
        while time.monotonic() < deadline:
            sample = latest['sample']
            if sample is None:
                print('Stopped: no camera frame')
                break
            frame, captured_at = sample
            now = time.monotonic()
            # Check freshness BEFORE skipping repeated frames.
            if now - captured_at > FRAME_TIMEOUT:
                print('Stopped: stale camera frame')
                break
            if captured_at == previous_stamp:
                time.sleep(0.005)
                continue
            previous_stamp = captured_at
            positions, overlay, _ = line_tracker.detect(frame, now, controller.preferred_direction())
            markers, green_mask = detect_green(frame)
            height, width = frame.shape[:2]
            markers = marker_tracker.update(markers, now, width, height)
            direction, marker_ids = marker_tracker.instruction(positions, markers, width, height)
            now = time.monotonic()
            if now >= deadline or now - captured_at > FRAME_TIMEOUT:
                print('Stopped: deadline or slow image processing')
                break
            left, right, state = controller.step(positions, direction, now, marker_ids, reliable=not line_tracker.ambiguous)
            marker_tracker.consume(controller.consumed_ids)
            if drive:
                robot.set_motors(left, right)
            if now >= next_display:
                points = get_line_points(positions, width, height, include_extra=True)
                for a,b in zip(points, points[1:]):
                    cv2.line(overlay, tuple(map(int,a)), tuple(map(int,b)), (255,0,255), 2)
                for marker in markers:
                    x,y,w,h = marker['box']
                    cv2.rectangle(overlay, (x,y), (x+w,y+h), (0,255,0), 2)
                    cv2.putText(overlay, str(marker['track_id']) + (' ready' if marker['eligible'] else ' ignored'), (x,max(10,y-3)), cv2.FONT_HERSHEY_SIMPLEX,.3,(0,255,255),1)
                overlay_view.value = bgr8_to_jpeg(overlay)
                mask_view.value = bgr8_to_jpeg(cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR))
                status_view.value = '%s | %s | marker=%s | L=%.2f R=%.2f' % (
                    'DRIVE' if drive else 'PREVIEW',state,direction,left,right) + ' | path=%.2f' % line_tracker.confidence
                next_display = now + 0.1
            if controller.stopped:
                print(state)
                break
            time.sleep(0.01)
    finally:
        robot.stop()
        print('Run ended; motors stopped.')

def run_preview(seconds=20):
    run_v7(seconds=seconds, drive=False)

def run_robot(seconds=2):
    run_v7(seconds=seconds, drive=True)


## 7. Stationary preview
Move the robot by hand through plain sharp corners, consecutive opposite bends, and left/right markers. Orange candidates should remain on tape and green selections should form a plausible route. Check CURVE_ALIGN / GREEN_ALIGN as the outgoing line appears. HOLD indicates competing paths, not a white gap. Green labels show visual IDs and eligibility. Preview uses the same deadlines and may stop; reposition and start another run.


In [ ]:
run_preview(seconds=20)


## 8. Short driving tests
First test unmarked single and consecutive bends, then one left/right marker, then close consecutive markers. Driving stays commented out below. Keep a physical stop accessible. The fourth (extra_far) band is enabled at 20–34% of image height. In stationary preview, check that this region sees floor tape rather than the horizon or background. Orange candidates and the magenta selected path now cover all four bands. Set EXTRA_FAR_ENABLED = False and rerun the definition cells to compare three-band behavior.

Tune motor speeds to reliable floor movement. Adjust GREEN_TRIGGER_Y for the pivot arc (smaller is earlier). If orange dots follow tape but green selections jump, inspect path association; do not compensate with motor gains. HOLD means the tracker cannot choose confidently. If a new marker is ignored, inspect its ID and whether it was first seen above the trigger. Fast jumps or disappearance longer than MARKER_MEMORY can break identity; this remains camera-only heuristic tracking.

TURN_TIMEOUT, LOST_TIMEOUT, GREEN_TURN_TIMEOUT and GAP_TIMEOUT are failure bounds, not turn-angle controls. Slow alignment may need measured adjustments, but never extend them just to hide a wrong-path failure. The controller cannot follow a route that stays outside the camera view indefinitely. V7 needs physical validation on your course.


In [ ]:
# Explicitly uncomment to move the robot:
# run_robot(seconds=2)


## 9. Cleanup
Run before switching notebooks. Shut down this kernel afterward.


In [ ]:
robot.stop()
camera.unobserve(receive_frame, names="value")
camera_link.unlink()
camera.stop()
